In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import interpolate
import datetime
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    days_in_year = (decimal_year - year) * 365
    date = pd.to_datetime(year, format='%Y') + pd.to_timedelta(days_in_year, unit='D')
    return date

In [ ]:
#data = pd.read_csv('samples_20251220.csv')
data = pd.read_csv('20260218/samples_20260218.csv')

data['path_name'] = data['sample'].str[0]
data['path_num'] = pd.to_numeric(data['sample'].str[2:-1])
data

In [ ]:
dark_start =  4/12 # which month the dark stripes start. For example for solomon april may.
dark_length = 2/3 # how long dry season spans (dense bands)

paths = pd.unique(data['path_name'])
res_array = np.array([])
res_array2 = np.array([])

s1 = data['age2'].rename('age')+dark_start
s2 = data['age_end'].rename('age')+dark_start+dark_length
data['age'] = s1.combine_first(s2)

for p in paths:

    data_iter = data[(data['path_name']==p)]
    data_iter_ = data[(data['path_name']==p) & (~data['age'].isna())]
    f  = interpolate.interp1d(data_iter_['path_num'], data_iter_['age'], bounds_error=False)
    res = f(data_iter['path_num'].values)
    res_array = np.append(res_array,res)

data['age_lin'] = res_array
data.iloc[70:]

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(data['path_num'], data['age_lin'], ".", label='model')
ax.plot(data['path_num'], data['age2']+dark_start, "x", label='dark start')
ax.plot(data['path_num'], data['age_end']+dark_start+dark_length, "x", label='dark end')
#ax.set_ylim(0, 10)
ax.legend()


In [ ]:
data.to_csv("20260218/interp_20260218.csv")

In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('20260218/interp_20260218.xlsx')
paths = list(df['path_name'].value_counts().index.values)
paths = np.flip(np.sort(paths))
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma']] = df_sorted[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).std()
df_sorted['d13c_no_tectonic'] = df_sorted['d13c'] - df_sorted['d13c_ma2'].fillna(0)
df_sorted['d18o_no_tectonic'] = df_sorted['d18o'] - df_sorted['d18o_ma2'].fillna(0)
df_sorted[['d13c_notec_ma','d18o_notec_ma']] = df_sorted[['d13c_no_tectonic','d18o_no_tectonic']].rolling(window_size, center=True).mean()


df_sorted

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df_sorted[df_sorted['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_absolute'], y=df_iter['d13c'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d13c'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_absolute'].rolling(window_size, center=True).mean(), y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))
    
fig.add_trace(go.Scatter(x=[1936+(31+29+31+19)/365], y=[-0.41], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))
fig.add_shape(type='line',
    x0=1936+(31+29+22)/365, x1=1936+(31+29+31+19)/365,
    y0=0, y1=1,  # Use yref='paper' for full height
    xref='x', yref='paper',
    line=dict(color="red", width=2, dash='dash'),
    name="EQ line"
)
fig.update_layout(width=1200, height=600, title_text="d13c", template='simple_white')
fig.show()

#fig.write_html('d18O_paths.html')

In [ ]:
blue = "#4287f5"
orange = "#f58442"
green = "#42f5ad"
red = "#f54254"
yellow = "#e4f000"
purple = "#f000ec"
green2 = "#00f070"
blue2 = "#00c0f0"
pink = "#e37b7b"
colors = [blue, orange, green, red, yellow, purple, green2, blue2, pink]



fig = go.Figure()
for i, path in enumerate(paths):
    df_iter = df_sorted[df_sorted['path_name']==path]
    fig.add_trace( go.Scatter(x=df_iter['age_absolute'], y=df_iter['d18o'], mode='markers', marker=dict(color=colors[i]), name=path, text=df_iter['Name']))
    path_ma = df_iter['d18o'].rolling(window_size, center=True).mean()
    fig.add_trace(go.Scatter(x=df_iter['age_absolute'].rolling(window_size, center=True).mean(), y=path_ma, mode='lines',marker=dict(color=colors[i]), showlegend=False))
    
#fig.add_trace(go.Scatter(x=[1936+(31+29+31+19)/365], y=[26.235], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='N Gizo EQ 1936 Apr'))
fig.add_shape(type='line',
    x0=1936+(31+29+22)/365, x1=1936+(31+29+31+19)/365,
    y0=0, y1=1,  # Use yref='paper' for full height
    xref='x', yref='paper',
    line=dict(color="red", width=2, dash='dash'),
    name="EQ line"
)
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
fig.show()
#fig.write_html('d18O_paths.html')